In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)

df.head()

In [ ]:
print("Dataset Information:\n")
print(df.info())

print("\nStatistical Summary:\n")
print(df.describe())

print("\nShape:", df.shape)

In [ ]:
print("Missing Values:\n")
print(df.isnull().sum())

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df.drop("Cabin", axis=1, inplace=True)

print("Missing Values After Preprocessing:\n")
print(df.isnull().sum())

In [ ]:
df.drop(["PassengerId", "Name", "Ticket"], axis=1, inplace=True)

df.head()

In [ ]:
encoder = LabelEncoder()

df["Sex"] = encoder.fit_transform(df["Sex"])
df["Embarked"] = encoder.fit_transform(df["Embarked"])

df.head()

In [ ]:
print("Duplicate Records:", df.duplicated().sum())

df = df.drop_duplicates()

print("New Dataset Shape:", df.shape)

In [ ]:
normalizer = MinMaxScaler()

cols = ["Age", "Fare", "SibSp", "Parch"]

df[cols] = normalizer.fit_transform(df[cols])

df.head()

In [ ]:
scaler = StandardScaler()

df[cols] = scaler.fit_transform(df[cols])

df.head()

In [ ]:
df["Fare_Log"] = np.log1p(df["Fare"] - df["Fare"].min() + 1)

df[["Fare", "Fare_Log"]].head()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(df.corr(), annot=True, cmap="coolwarm")

plt.title("Correlation Heatmap 24EU01130")
plt.show()

In [ ]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
pca = PCA()

X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio:\n")
print(pca.explained_variance_ratio_)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(pca.explained_variance_ratio_) + 1),
    np.cumsum(pca.explained_variance_ratio_),
    marker='o'
)

plt.xlabel("Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance 24EU01130")
plt.grid(True)

plt.show()

In [ ]:
pca = PCA(n_components=2)

X_new = pca.fit_transform(X_scaled)

print("Shape after PCA:", X_new.shape)

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X, y)

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)

print(importance)

In [ ]:
X_chi = df.drop("Survived", axis=1)

X_chi = X_chi - X_chi.min()

selector = SelectKBest(score_func=chi2, k=5)

X_selected = selector.fit_transform(X_chi, y)

print("Selected Features:")
print(X_chi.columns[selector.get_support()])

In [ ]:
df.to_csv("Titanic_Preprocessed.csv", index=False)

print("Dataset Saved Successfully.")

In [ ]:
from google.colab import files

files.download("Titanic_Preprocessed.csv")